# Dense index → full-text search: the whole migration, one phase per cell

This notebook runs every phase of `README.md` against **throwaway indexes**, with a background
workload writing to the source index the entire time — so you can watch how the pieces fit
together before pointing any of it at production.

What it costs: two serverless indexes for a few minutes, and a few thousand writes.

**Before you start**

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env            # add PINECONE_API_KEY
cp config.example.yaml config.yaml
set -a; source .env; set +a
jupyter lab
```

In [ ]:
import json
import time
from pathlib import Path

from fts_migrate import dense_source, importer, reconcile, simulate, target_index
from fts_migrate.cdc import CdcLog, CdcWrappedIndex, apply_changes
from fts_migrate.config import load_settings
from fts_migrate.convert import DocumentMapper, MissingTextError, iter_jsonl_dir, parquet_to_jsonl
from fts_migrate.cutover import DONE, RAMP, SHADOW, CutoverState, SearchRouter

settings = load_settings("config.yaml")
pc = dense_source.connect(settings)

export_dir = settings.export.dir / settings.source.namespace
jsonl_dir = settings.convert.dir / settings.target.namespace
CURSOR = "target"

print(f"source: {settings.source.index}/{settings.source.namespace}")
print(f"target: {settings.target.index}/{settings.target.namespace}")

## Phase 0 — A dense index that is being read and written

A real migration skips this: you already have the index. Here we create one and seed it with
records whose metadata carries the text we will later make searchable.

In [ ]:
simulate.ensure_demo_index(pc, settings)
source = dense_source.open_index(pc, settings.source.index)

written = simulate.seed_index(source, settings, settings.demo.records)
print(f"seeded {written} records")

time.sleep(10)
spec = dense_source.describe_source(pc, settings.source.index)
print(spec)

## Phase 1 — Turn on change capture, **before** the export

This ordering is the whole trick. Bulk import can only create namespaces that do not yet
exist, so the target namespace cannot be written to until the import finishes. Everything
that lands in between has to be buffered.

`CdcWrappedIndex` is a drop-in for the index object your application already writes through:
same writes to the dense index, plus an append to a durable log.

In [ ]:
log = CdcLog(settings.cdc.db)
wrapped = CdcWrappedIndex(source, log, namespace=settings.source.namespace)

workload = simulate.Workload(wrapped, settings, rate_per_second=settings.demo.write_rate)
workload.prime(list(dense_source.iter_ids(source, settings.source.namespace, limit=500)))
workload.start()

print(f"workload writing at {settings.demo.write_rate}/s; CDC head seq {log.head_seq()}")

## Phase 2 — Export to Parquet

The real path is a [backup export](https://docs.pinecone.io/guides/manage-data/export-backup):
create a backup, ask Support to export it to your bucket, and you get Parquet in exactly this
shape — `id`, `values`, `metadata` as a JSON string. Here we write those files ourselves.

Note the `snapshot_seq` in the manifest: it records which CDC changes the export predates.

In [ ]:
manifest = simulate.export_namespace(
    source,
    settings.source.namespace,
    export_dir,
    rows_per_file=settings.export.rows_per_file,
    include_text=True,
    text_key=settings.source.text_metadata_key,
    snapshot_seq=log.head_seq(),
    source_index=settings.source.index,
)
print(json.dumps({k: v for k, v in manifest.items() if k != "files"}, indent=2))

## Phase 3 — Convert Parquet to JSONL

Document-schema imports read JSON Lines, not Parquet. The mapping is `id` → `_id`, `values` →
the schema's dense field, and the `metadata` JSON string spread into top-level fields.

Conversion also enforces the document-API limits locally — 2 MB per document, 100 KB and
10,000 tokens per full-text field, 40 KB of metadata, field names not starting with `_` or `$`
— so problems surface here rather than as per-row errors part-way through an import.

In [ ]:
mapper = DocumentMapper(settings, dimension=spec.dimension)
stats = parquet_to_jsonl(
    export_dir, jsonl_dir, mapper, use_gzip=settings.convert.gzip,
    rows_per_file=settings.export.rows_per_file,
)
print(stats.summary())

first = next(iter_jsonl_dir(jsonl_dir))
preview = {k: (v[:3] if k == settings.target.dense_field else v) for k, v in first.items()}
print(json.dumps(preview, indent=2)[:600])

### What happens when the export has no text

Plenty of exports carry vectors and metadata but not the searchable text — it lives in the
system of record. Conversion stops, loudly, naming the field and the row. That is deliberate:
an index whose full-text field is empty cannot do BM25, and a silent load would leave you with
an index that looks healthy and ranks nothing.

There is nothing to invent here — join the text back in, keyed by record id, before converting.

In [ ]:
no_text_dir = Path("work/export-no-text") / settings.source.namespace
simulate.export_namespace(
    source, settings.source.namespace, no_text_dir,
    include_text=False, text_key=settings.source.text_metadata_key,
    source_index=settings.source.index,
)

try:
    parquet_to_jsonl(
        no_text_dir, Path("work/jsonl-no-text"), DocumentMapper(settings, dimension=spec.dimension)
    )
except MissingTextError as exc:
    print("stopped, as it should:\n")
    print(exc)

## Phase 4 — Create the target index

The schema is fixed at creation — fields cannot be added, removed or retyped afterwards. The
dense field's dimension and metric are copied from the source index so they cannot drift.

In [ ]:
schema = target_index.build_schema(settings, spec)
print(json.dumps(schema, indent=2))

model = target_index.create_index(pc, settings, spec)
target = target_index.open_index(pc, settings)
print(f"ready: {getattr(model, 'host', '?')}")

## Phase 5 — Load the documents

`import` mode uploads the JSONL to object storage and runs a real bulk import — the path a
production migration takes, and one that takes at least ten minutes. `upsert` mode streams the
same documents through the documents API, which needs no bucket and is what this notebook
uses.

Either way the load finishes with a freshness probe: documents index asynchronously, and
replaying a delete before the document it deletes has landed is a delete that does nothing.

In [ ]:
loaded = importer.upsert_from_jsonl(target, settings.target.namespace, jsonl_dir)
print(f"loaded {loaded} documents")

sample_ids = [doc["_id"] for _, doc in zip(range(100), iter_jsonl_dir(jsonl_dir))]
print("fetchable:", importer.wait_until_searchable(target, settings.target.namespace, sample_ids))

### The bulk-import path, for reference

```python
prefix = importer.upload_tree(jsonl_dir, settings.import_.uri, settings.target.namespace)
import_id = importer.start_import(
    target, settings.import_.uri,
    integration_id=settings.import_.integration_id,
    error_mode=settings.import_.error_mode,
)
state = importer.wait_for_import(target, import_id, on_poll=print)
```

## Phase 6 — Replay everything that landed during the load

The workload has been writing this whole time. Stop it, then drain the log.

Replay reduces the window to one write per document. An upsert or a delete says everything
about a document on its own; a partial `update` does not, so a document whose last change is a
patch is replayed from its full history — folded into the upsert behind it, or sent as a
`documents.update` patch if the document came from the bulk load.

That makes replay idempotent: running it twice applies the same end state.

In [ ]:
workload_stats = workload.stop()
print(workload_stats.summary())
print("CDC lag before replay:", log.lag(CURSOR))

replay_mapper = DocumentMapper(settings, dimension=spec.dimension, allow_missing_text=True)
print(apply_changes(target, log, replay_mapper, namespace=settings.target.namespace,
                    cursor_name=CURSOR, source_namespace=settings.source.namespace).summary())

time.sleep(20)
print(apply_changes(target, log, replay_mapper, namespace=settings.target.namespace,
                    cursor_name=CURSOR, source_namespace=settings.source.namespace).summary())
print("CDC lag after replay:", log.lag(CURSOR))

## Phase 7 — Prove the two indexes agree

Four checks, weakest to strongest. Counts can agree while the wrong documents are present, so
the id diff and query parity are what sign-off actually rests on.

An **orphan** — a document on the target the source no longer has — matters as much as a
missing one: it is what a dropped delete looks like.

In [ ]:
time.sleep(20)

counts = reconcile.compare_counts(source, target, settings)
print(counts.render())

ids = reconcile.diff_ids(source, target, settings)
print(ids.render())

sample = list(dense_source.iter_ids(source, settings.source.namespace, limit=100))
fields = reconcile.diff_fields(source, target, settings, sample)
print(fields.render())

parity = reconcile.query_parity(source, target, settings, queries=10, top_k=10)
print(parity.render())

### The reason you did all this

A keyword query the dense index could never have answered.

In [ ]:
for doc_id, score in reconcile.bm25_probe(target, settings, "bm25 relevance ranking", top_k=5):
    print(f"{doc_id}  {score:.4f}")

phrase = target.documents.search(
    namespace=settings.target.namespace,
    top_k=3,
    score_by=[{"type": "query_string", "query": "text:(bm25 AND ranking) NOT invoice"}],
    include_fields=settings.target.text_fields,
)
print("\nLucene query_string:", [m.id for m in phrase.matches])

## Phase 8 — Cut over, reversibly

Reads move in stages while writes keep going to both indexes. In `shadow` the dense index
answers and the new one is queried alongside for comparison only, so a mismatch is counted
without ever reaching a user. Rollback is setting the percentage back to 0.

In [ ]:
state = CutoverState(mode=SHADOW)
router = SearchRouter(source, target, settings, state, seed=1)

probe = list(dense_source.fetch_records(source, sample[:5], settings.source.namespace).values())
for record in probe:
    router.search(record["values"], top_k=10)
print("shadow:", router.stats.render())

router.state.mode, router.state.target_read_pct = RAMP, 50.0
for record in probe:
    router.search(record["values"], top_k=10)
print("ramp:  ", router.stats.render())

router.state.mode = DONE
print("keyword search, target only:", router.search_text("vector database", top_k=3))

router.state.save(Path("work/cutover.json"))

## Clean up

Deletes both throwaway indexes and the CDC log. Skip this if you want to keep poking at the
new index.

In [ ]:
log.close()
for name in (settings.source.index, settings.target.index):
    if pc.has_index(name):
        pc.delete_index(name)
        print(f"deleted {name}")
if settings.cdc.db.exists():
    settings.cdc.db.unlink()